<h2>Description</h2>

L'objectif de ce code est de fusionner toutes les données, donc à savoir les données de la rue de la convention avec les données externes. 
Nous voulons donc créer un dataframe qui contient :
<li>les informations météorologiques</li>
<li>les informations sur les vacances et jour fériés</li>
<li>les informations sur l'opération "Paris respire"</li>

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

doc = 'convention.csv'

print("Version installée de pandas : ")
print(pd.__version__)

Version installée de pandas : 
2.3.0


<h3>Analyse du dataset relatif à l'axe</h3>

In [2]:
df_axe = pd.read_csv("../datasets_axes_bruts/" + doc, sep=";")

# On convertit la date en format convenable 
df_axe['Date et heure de comptage'] = pd.to_datetime(df_axe['Date et heure de comptage'], 
                                      errors='coerce', utc=True).dt.tz_convert('Europe/Paris').dt.tz_localize(None)  

df_axe.head() 

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,Etat arc,Date debut dispo data,Date fin dispo data,geo_point_2d,geo_shape
0,5671,Convention,2025-09-02 11:00:00,462.0,4.20056,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,20/01/1997,01/01/2023,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
1,5671,Convention,2025-10-09 11:00:00,514.0,3.86722,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,20/01/1997,01/01/2023,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
2,5671,Convention,2025-09-02 12:00:00,482.0,3.24945,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,20/01/1997,01/01/2023,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
3,5671,Convention,2025-09-02 13:00:00,563.0,4.57445,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,20/01/1997,01/01/2023,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
4,5671,Convention,2024-10-04 12:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,Invalide,20/01/1997,01/01/2023,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."


In [3]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval
count,9266.0,9266,4685.000000,4792.000000,9266.0,9266.0
mean,5671.0,2025-04-05 02:51:17.055903488,398.531057,3.590080,2937.0,2973.0
min,5671.0,2024-09-01 05:00:00,31.000000,0.000000,2937.0,2973.0
25%,5671.0,2024-12-27 11:15:00,196.000000,1.310973,2937.0,2973.0
50%,5671.0,2025-04-06 00:30:00,389.000000,2.623890,2937.0,2973.0
75%,5671.0,2025-07-24 11:45:00,578.000000,4.442778,2937.0,2973.0
max,5671.0,2025-10-30 00:00:00,1024.000000,37.348340,2937.0,2973.0
std,0.0,NaN,225.331508,3.744822,0.0,0.0


In [4]:
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9266 entries, 0 to 9265
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            9266 non-null   int64         
 1   Libelle                    9266 non-null   object        
 2   Date et heure de comptage  9266 non-null   datetime64[ns]
 3   Débit horaire              4685 non-null   float64       
 4   Taux d'occupation          4792 non-null   float64       
 5   Etat trafic                9266 non-null   object        
 6   Identifiant noeud amont    9266 non-null   int64         
 7   Libelle noeud amont        9266 non-null   object        
 8   Identifiant noeud aval     9266 non-null   int64         
 9   Libelle noeud aval         9266 non-null   object        
 10  Etat arc                   9266 non-null   object        
 11  Date debut dispo data      9266 non-null   object        
 12  Date f

In [5]:
df_axe['date'] = df_axe['Date et heure de comptage'].dt.date
df_axe['heure'] = df_axe['Date et heure de comptage'].dt.hour
df_axe['dow'] = df_axe['Date et heure de comptage'].dt.dayofweek
df_axe['mois'] = df_axe['Date et heure de comptage'].dt.month
df_axe['annee'] = df_axe['Date et heure de comptage'].dt.year
df_axe['jour_mois'] = df_axe['Date et heure de comptage'].dt.day
jour_map = {0:'Lun',1:'Mar',2:'Mer',3:'Jeu',4:'Ven',5:'Sam',6:'Dim'}
df_axe['jour_semaine'] = df_axe['dow'].map(jour_map)

In [6]:
fig = px.line(
    df_axe.sort_values('Date et heure de comptage'),
    x='Date et heure de comptage', y='Débit horaire',
    title=f"Débit horaire",
    labels={'Date et heure de comptage':'Date/Heure','Débit horaire':'Débit (véh/h)'}
)

fig.show()

In [7]:
profil = (df_axe
          .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
          .mean())

fig = px.line(
    profil, x='heure', y='Débit horaire', color='jour_semaine',
    markers=True, title=f"Profil horaire moyen par jour",
    labels={'heure':'Heure','Débit horaire':'Débit moyen (véh/h)','jour_semaine':'Jour'}
)
fig.update_layout(template='simple_white')
fig.show()

In [8]:
heat = (df_axe
        .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
        .mean())


heat['jour_semaine'] = pd.Categorical(heat['jour_semaine'],
                                      categories=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
                                      ordered=True)

fig = px.imshow(
    heat.pivot(index='jour_semaine', columns='heure', values='Débit horaire').values,
    labels=dict(x="Heure", y="Jour", color="Débit moyen"),
    x=list(range(24)),
    y=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
    title=f"Carte de chaleur"
)
fig.update_layout(template='simple_white')
fig.show()


In [9]:
df_scatter = df_axe.dropna(subset=['Débit horaire','Taux d\'occupation']).copy()
fig = px.scatter(
    df_scatter, x='Taux d\'occupation', y='Débit horaire',
    color='jour_semaine', opacity=0.7,
    title=f"Débit vs Taux d’occupation",
    labels={'Taux d\'occupation':'Taux d’occupation','Débit horaire':'Débit (véh/h)'}
)

fig.update_layout(template='simple_white')
fig.show()

In [10]:
box = df_axe.dropna(subset=['Débit horaire']).copy()
fig = px.box(
    box, x='jour_semaine', y='Débit horaire', points='all',
    title=f"Distribution du débit par jour",
    labels={'jour_semaine':'Jour','Débit horaire':'Débit (véh/h)'}
)
fig.update_layout(template='simple_white')
fig.show()


In [11]:
etat_jour = (df_axe
             .assign(jour=pd.to_datetime(df_axe['date']))
             .groupby(['jour','Etat trafic'], as_index=False)
             .size())

fig = px.area(
    etat_jour, x='jour', y='size', color='Etat trafic',
    title=f"État du trafic (comptes journaliers)",
    labels={'jour':'Date','size':'Occurrences'}
)
fig.update_layout(template='simple_white', hovermode='x unified')
fig.show()

In [12]:
mensuel = (df_axe
           .set_index('Date et heure de comptage')
           .resample('MS')['Débit horaire']
           .mean()
           .to_frame('debit_moyen')
           .reset_index())

mensuel['trend_3m'] = mensuel['debit_moyen'].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['debit_moyen'],
    mode='lines+markers', name='Moyenne mensuelle'
))
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['trend_3m'],
    mode='lines', name='Tendance (MM-3)', line=dict(width=4)
))

fig.update_layout(
    title="Débit horaire – tendance mensuelle (moyenne + lissage 3 mois)",
    xaxis_title="Mois",
    yaxis_title="Débit moyen (véh/h)",
    template="simple_white",
    hovermode="x unified"
)
fig.show()


In [13]:
profil = (df_axe
          .groupby(['annee','mois'], as_index=False)['Débit horaire']
          .mean()
          .rename(columns={'Débit horaire':'debit_moyen'}))

fig = px.line(
    profil, x='mois', y='debit_moyen', color='annee',
    markers=True,
    title="Profil mensuel du débit par année",
    labels={'mois':'Mois', 'debit_moyen':'Débit moyen (véh/h)', 'annee':'Année'}
)
fig.update_layout(template='simple_white', xaxis=dict(dtick=1))
fig.show()


In [14]:
df_axe['heure_sin'] = np.sin(2 * np.pi * df_axe['heure'] / 24)
df_axe['heure_cos'] = np.cos(2 * np.pi * df_axe['heure'] / 24)

df_axe['jour_sin'] = np.sin(2 * np.pi * df_axe['dow'] / 7)
df_axe['jour_cos'] = np.cos(2 * np.pi * df_axe['dow'] / 7)

df_axe['mois_sin'] = np.sin(2 * np.pi * df_axe['mois'] / 12)
df_axe['mois_cos'] = np.cos(2 * np.pi * df_axe['mois'] / 12)

In [15]:
df_axe.head(10)

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,mois,annee,jour_mois,jour_semaine,heure_sin,heure_cos,jour_sin,jour_cos,mois_sin,mois_cos
0,5671,Convention,2025-09-02 11:00:00,462.0,4.20056,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,9,2025,2,Mar,2.588190e-01,-0.965926,0.781831,0.623490,-1.000000e+00,-1.836970e-16
1,5671,Convention,2025-10-09 11:00:00,514.0,3.86722,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,10,2025,9,Jeu,2.588190e-01,-0.965926,0.433884,-0.900969,-8.660254e-01,5.000000e-01
2,5671,Convention,2025-09-02 12:00:00,482.0,3.24945,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,9,2025,2,Mar,1.224647e-16,-1.000000,0.781831,0.623490,-1.000000e+00,-1.836970e-16
3,5671,Convention,2025-09-02 13:00:00,563.0,4.57445,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,9,2025,2,Mar,-2.588190e-01,-0.965926,0.781831,0.623490,-1.000000e+00,-1.836970e-16
4,5671,Convention,2024-10-04 12:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,10,2024,4,Ven,1.224647e-16,-1.000000,-0.433884,-0.900969,-8.660254e-01,5.000000e-01
5,5671,Convention,2025-02-18 21:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-7.071068e-01,0.707107,0.781831,0.623490,8.660254e-01,5.000000e-01
6,5671,Convention,2025-02-18 20:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-8.660254e-01,0.500000,0.781831,0.623490,8.660254e-01,5.000000e-01
7,5671,Convention,2025-02-18 19:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-9.659258e-01,0.258819,0.781831,0.623490,8.660254e-01,5.000000e-01
8,5671,Convention,2025-02-18 15:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-7.071068e-01,-0.707107,0.781831,0.623490,8.660254e-01,5.000000e-01
9,5671,Convention,2024-12-09 01:00:00,157.0,0.77889,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,12,2024,9,Lun,2.588190e-01,0.965926,0.000000,1.000000,-2.449294e-16,1.000000e+00


<h3>Nous insérons d'abord les données météorologiques</h3>

In [16]:
meteo = pd.read_csv("../datasets_externes_clean/meteo.csv", sep=";")
meteo['datetime'] = pd.to_datetime(meteo['datetime'])

In [17]:
df_merge = pd.merge(df_axe, meteo, left_on="Date et heure de comptage", right_on="datetime", how="left")

In [18]:
df_merge.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,jour_cos,mois_sin,mois_cos,Unnamed: 0,precipitations heure,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),datetime
count,9266.0,9266,4685.000000,4792.000000,9266.0,9266.0,9266.000000,9266.000000,9266.000000,9266.000000,...,9266.000000,9.266000e+03,9.266000e+03,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266
mean,5671.0,2025-04-05 02:51:17.055903488,398.531057,3.590080,2937.0,2973.0,11.505720,2.991150,6.762789,2024.738291,...,0.000939,-1.163672e-01,1.350755e-02,11042.854738,0.084783,4.022232,2.938431,13.447151,13.021261,2025-04-05 02:51:17.055903488
min,5671.0,2024-09-01 05:00:00,31.000000,0.000000,2937.0,2973.0,0.000000,0.000000,1.000000,2024.000000,...,-0.900969,-1.000000e+00,-1.000000e+00,5861.000000,0.000000,0.000000,0.000000,-3.600000,0.000000,2024-09-01 05:00:00
25%,5671.0,2024-12-27 11:15:00,196.000000,1.310973,2937.0,2973.0,6.000000,1.000000,4.000000,2024.000000,...,-0.900969,-8.660254e-01,-5.000000e-01,8675.250000,0.000000,0.000000,2.000000,9.100000,0.000000,2024-12-27 11:15:00
50%,5671.0,2025-04-06 00:30:00,389.000000,2.623890,2937.0,2973.0,12.000000,3.000000,7.000000,2025.000000,...,-0.222521,-2.449294e-16,-1.836970e-16,11064.500000,0.000000,0.000000,2.800000,13.400000,0.000000,2025-04-06 00:30:00
75%,5671.0,2025-07-24 11:45:00,578.000000,4.442778,2937.0,2973.0,17.000000,5.000000,10.000000,2025.000000,...,0.623490,5.000000e-01,5.000000e-01,13691.750000,0.000000,0.000000,3.700000,17.800000,20.000000,2025-07-24 11:45:00
max,5671.0,2025-10-30 00:00:00,1024.000000,37.348340,2937.0,2973.0,23.000000,6.000000,12.000000,2025.000000,...,1.000000,1.000000e+00,1.000000e+00,16032.000000,14.300000,60.000000,9.300000,37.600000,60.000000,2025-10-30 00:00:00
std,0.0,NaN,225.331508,3.744822,0.0,0.0,6.920146,2.002354,3.345876,0.439589,...,0.708282,7.503930e-01,6.506097e-01,2955.156175,0.541102,12.929655,1.314630,6.599812,21.858906,NaN


In [19]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9266 entries, 0 to 9265
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            9266 non-null   int64         
 1   Libelle                    9266 non-null   object        
 2   Date et heure de comptage  9266 non-null   datetime64[ns]
 3   Débit horaire              4685 non-null   float64       
 4   Taux d'occupation          4792 non-null   float64       
 5   Etat trafic                9266 non-null   object        
 6   Identifiant noeud amont    9266 non-null   int64         
 7   Libelle noeud amont        9266 non-null   object        
 8   Identifiant noeud aval     9266 non-null   int64         
 9   Libelle noeud aval         9266 non-null   object        
 10  Etat arc                   9266 non-null   object        
 11  Date debut dispo data      9266 non-null   object        
 12  Date f

In [20]:
df_axe = df_merge

<h3>Nous allons maintenant ajouter les données relatives au jours piétonnisés</h3>

Nous allons donc ajouter une colonne de 1 ou 0 pour indiquer si oui ou non il s'agissait d'un jour piéton.

In [21]:
pieton = pd.read_csv("../datasets_externes_clean/pieton.csv", sep=";")
pieton.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  18 non-null     int64 
 1   date_debut  18 non-null     object
 2   date_fin    18 non-null     object
dtypes: int64(1), object(2)
memory usage: 560.0+ bytes


In [22]:
pieton['date_debut'] = pd.to_datetime(pieton['date_debut'])
pieton['date_fin'] = pd.to_datetime(pieton['date_fin'])

Nous écrivons ci-dessous une fonction pour déterminer si pour une date donnée, certains axes parisiens étaient piétonnisés.

In [23]:
def est_pietonnise(date):
    debut = pieton['date_debut']
    fin = pieton['date_fin']
    for i in range (len(debut)):
        if date >= debut[i] and date < fin[i]:
            return 1
    return 0

df_axe['est_pieton'] = df_axe['Date et heure de comptage'].apply(est_pietonnise)

In [24]:
df_axe[df_axe['est_pieton'] == 1][['Date et heure de comptage', 'est_pieton']]

,Date et heure de comptage,est_pieton
231,2024-11-03 16:00:00,1
232,2024-11-03 15:00:00,1
240,2024-11-03 14:00:00,1
241,2024-11-03 13:00:00,1
242,2024-11-03 12:00:00,1
...,...,...
6761,2025-05-04 17:00:00,1
6762,2025-05-04 16:00:00,1
6763,2025-05-04 15:00:00,1
6764,2025-05-04 13:00:00,1


In [25]:
df_axe = df_axe.sort_values(by="Date et heure de comptage", ascending=True)

<h3>On va maintenant ajouter des données sur les vacances, jours fériés...</h3>

In [26]:
vacances = pd.read_csv("../datasets_externes_clean/vacances.csv", sep=";")
vacances ['Date de début'] = pd.to_datetime(vacances['Date de début'])
vacances ['Date de fin'] = pd.to_datetime(vacances['Date de fin'])

In [27]:
def est_jour_vacances(dat):
    date_sans_heure = dat.date()
    debut = vacances['Date de début'].dt.date
    fin = vacances['Date de fin'].dt.date
    for i in range (len(debut)):
        if date_sans_heure >= debut[i] and date_sans_heure < fin[i]:
            return 1
    return 0

df_axe['est_vacances'] = df_axe['Date et heure de comptage'].apply(est_jour_vacances)

In [28]:
df = df_axe[df_axe['est_vacances'] == 1][['Date et heure de comptage', 'est_vacances']]

In [29]:
df.head(50)

,Date et heure de comptage,est_vacances
2525,2024-09-01 05:00:00,1
2524,2024-09-01 06:00:00,1
2479,2024-09-01 07:00:00,1
2478,2024-09-01 08:00:00,1
2523,2024-09-01 09:00:00,1
2477,2024-09-01 10:00:00,1
2522,2024-09-01 11:00:00,1
2521,2024-09-01 12:00:00,1
2520,2024-09-01 13:00:00,1
2519,2024-09-01 14:00:00,1


In [30]:
def est_avant_vacances(date):
    date_sans_heure = date.date()
    debut = vacances['Date de début'].dt.date
    for i in range (len(debut)):
        if date_sans_heure == debut[i] - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_vacances'] = df_axe['Date et heure de comptage'].apply(est_avant_vacances)

In [31]:
df_axe[df_axe['est_avant_vacances'] == 1][['est_avant_vacances', 'Date et heure de comptage']]

,est_avant_vacances,Date et heure de comptage
6707,1,2024-10-18 00:00:00
6510,1,2024-10-18 01:00:00
6186,1,2024-10-18 02:00:00
6509,1,2024-10-18 03:00:00
6185,1,2024-10-18 04:00:00
...,...,...
4358,1,2025-10-17 19:00:00
4359,1,2025-10-17 20:00:00
4134,1,2025-10-17 21:00:00
4360,1,2025-10-17 22:00:00


In [32]:
ferie = pd.read_csv("../datasets_externes_clean/ferie.csv", sep=";")

ferie['Date de début'] = pd.to_datetime(ferie['Date de début'], format='%d/%m/%Y')

In [33]:
def est_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el:
            return 1
    return 0

df_axe['est_ferie'] = df_axe['Date et heure de comptage'].apply(est_ferie)

In [34]:
df_axe[df_axe['est_ferie']==1]['Date et heure de comptage']

9247   2024-11-01 00:00:00
9130   2024-11-01 01:00:00
9265   2024-11-01 02:00:00
9264   2024-11-01 03:00:00
3199   2024-11-01 04:00:00
               ...        
6777   2025-08-15 19:00:00
7214   2025-08-15 20:00:00
6776   2025-08-15 21:00:00
7213   2025-08-15 22:00:00
6775   2025-08-15 23:00:00
Name: Date et heure de comptage, Length: 240, dtype: datetime64[ns]

In [35]:
def est_avant_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_ferie'] = df_axe['Date et heure de comptage'].apply(est_avant_ferie)

In [36]:
df_axe[df_axe['est_avant_ferie']==1]['Date et heure de comptage']

9163   2024-10-31 00:00:00
9258   2024-10-31 01:00:00
9257   2024-10-31 02:00:00
9243   2024-10-31 03:00:00
9242   2024-10-31 04:00:00
               ...        
6349   2025-08-14 19:00:00
6804   2025-08-14 20:00:00
6350   2025-08-14 21:00:00
6351   2025-08-14 22:00:00
6352   2025-08-14 23:00:00
Name: Date et heure de comptage, Length: 240, dtype: datetime64[ns]

In [37]:
df_axe['est_weekend'] = df_axe['dow'].isin([5, 6]).astype(int)

Nous vérifions maintenant la composition du dataset

In [38]:
df_axe = df_axe.drop('Unnamed: 0', axis = 1)
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9266 entries, 2525 to 8977
Data columns (total 41 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            9266 non-null   int64         
 1   Libelle                    9266 non-null   object        
 2   Date et heure de comptage  9266 non-null   datetime64[ns]
 3   Débit horaire              4685 non-null   float64       
 4   Taux d'occupation          4792 non-null   float64       
 5   Etat trafic                9266 non-null   object        
 6   Identifiant noeud amont    9266 non-null   int64         
 7   Libelle noeud amont        9266 non-null   object        
 8   Identifiant noeud aval     9266 non-null   int64         
 9   Libelle noeud aval         9266 non-null   object        
 10  Etat arc                   9266 non-null   object        
 11  Date debut dispo data      9266 non-null   object        
 12  Date fin

In [39]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,force moyenne vent (m/s),Température,ensoleillement (en min),datetime,est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_weekend
count,9266.0,9266,4685.000000,4792.000000,9266.0,9266.0,9266.000000,9266.000000,9266.000000,9266.000000,...,9266.000000,9266.000000,9266.000000,9266,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000
mean,5671.0,2025-04-05 02:51:17.055903232,398.531057,3.590080,2937.0,2973.0,11.505720,2.991150,6.762789,2024.738291,...,2.938431,13.447151,13.021261,2025-04-05 02:51:17.055903232,0.010576,0.328405,0.015649,0.025901,0.025901,0.283510
min,5671.0,2024-09-01 05:00:00,31.000000,0.000000,2937.0,2973.0,0.000000,0.000000,1.000000,2024.000000,...,0.000000,-3.600000,0.000000,2024-09-01 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5671.0,2024-12-27 11:15:00,196.000000,1.310973,2937.0,2973.0,6.000000,1.000000,4.000000,2024.000000,...,2.000000,9.100000,0.000000,2024-12-27 11:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5671.0,2025-04-06 00:30:00,389.000000,2.623890,2937.0,2973.0,12.000000,3.000000,7.000000,2025.000000,...,2.800000,13.400000,0.000000,2025-04-06 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,5671.0,2025-07-24 11:45:00,578.000000,4.442778,2937.0,2973.0,17.000000,5.000000,10.000000,2025.000000,...,3.700000,17.800000,20.000000,2025-07-24 11:45:00,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,5671.0,2025-10-30 00:00:00,1024.000000,37.348340,2937.0,2973.0,23.000000,6.000000,12.000000,2025.000000,...,9.300000,37.600000,60.000000,2025-10-30 00:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,0.0,NaN,225.331508,3.744822,0.0,0.0,6.920146,2.002354,3.345876,0.439589,...,1.314630,6.599812,21.858906,NaN,0.102301,0.469658,0.124118,0.158849,0.158849,0.450726


In [40]:
df_axe.to_csv("../datasets_axes_with_all_features/" + doc , sep=";")